# EVolvAI: Generative Counterfactual Framework

This notebook trains the TCN-VAE on Google Colab with GPU acceleration.

It is a self-contained copy of the `generative_model/` package.


In [ ]:
!pip install torch numpy

In [ ]:
# ─── CONFIG ──────────────────────────────────────────────────────────────────
NUM_NODES       = 50
SEQ_LEN         = 24
NUM_FEATURES    = NUM_NODES + 1
NUM_SAMPLES     = 1000
BATCH_SIZE      = 32
TCN_CHANNELS    = [32, 64]
KERNEL_SIZE     = 2
DROPOUT         = 0.2
LATENT_DIM      = 16
COND_DIM        = 2
DECODER_HIDDEN  = 128
LEARNING_RATE   = 1e-3
EPOCHS          = 10
KLD_WEIGHT      = 1.0
BASELINE_CONDITION = [0.0, 1.0]
SCENARIOS = {
    "extreme_winter_storm": {"description": "Extreme winter storm + 2.5x fleet electrification", "condition": [1.0, 2.5]},
    "summer_peak":          {"description": "High summer temps + 1.5x electrification",          "condition": [0.5, 1.5]},
    "full_electrification": {"description": "Normal weather + 3.0x full fleet electrification",  "condition": [0.0, 3.0]},
}


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np

In [ ]:
# ─── DATA LOADER ─────────────────────────────────────────────────────────────
class EVDemandDataset(Dataset):
    def __init__(self, num_samples=NUM_SAMPLES, num_nodes=NUM_NODES, seq_len=SEQ_LEN):
        self.num_samples = num_samples
        base = np.random.uniform(10, 100, (num_samples, seq_len, num_nodes))
        diurnal = np.clip([1 + np.sin((h - 12) * np.pi / 12) for h in range(seq_len)], 0.5, 2.0).reshape(1, seq_len, 1)
        charge = (base * diurnal).astype(np.float32)
        weather = np.random.uniform(-10, 40, (num_samples, seq_len, 1)).astype(np.float32)
        charge  = (charge  - charge.mean())  / (charge.std()  + 1e-5)
        weather = (weather - weather.mean()) / (weather.std() + 1e-5)
        self.data = np.concatenate([charge, weather], axis=-1)
    def __len__(self):
        return self.num_samples
    def __getitem__(self, idx):
        return torch.from_numpy(self.data[idx])

def get_dataloader(batch_size=BATCH_SIZE, num_nodes=NUM_NODES):
    return DataLoader(EVDemandDataset(num_nodes=num_nodes), batch_size=batch_size, shuffle=True)


In [ ]:
# ─── MODELS ──────────────────────────────────────────────────────────────────
class CausalConv1d(nn.Conv1d):
    def __init__(self, in_channels, out_channels, kernel_size, stride=1, dilation=1, groups=1, bias=True):
        self._causal_padding = (kernel_size - 1) * dilation
        super().__init__(in_channels, out_channels, kernel_size=kernel_size, stride=stride,
                         padding=self._causal_padding, dilation=dilation, groups=groups, bias=bias)
    def forward(self, x):
        out = super().forward(x)
        return out[:, :, :-self._causal_padding] if self._causal_padding != 0 else out

class TCNBlock(nn.Module):
    def __init__(self, n_in, n_out, ks, stride, dilation, dropout=DROPOUT):
        super().__init__()
        self.net = nn.Sequential(
            CausalConv1d(n_in, n_out, ks, stride=stride, dilation=dilation), nn.ReLU(), nn.Dropout(dropout),
            CausalConv1d(n_out, n_out, ks, stride=stride, dilation=dilation), nn.ReLU(), nn.Dropout(dropout),
        )
        self.downsample = nn.Conv1d(n_in, n_out, 1) if n_in != n_out else None
        self.relu = nn.ReLU()
    def forward(self, x):
        res = x if self.downsample is None else self.downsample(x)
        return self.relu(self.net(x) + res)

class TemporalConvNet(nn.Module):
    def __init__(self, num_inputs, num_channels, kernel_size=KERNEL_SIZE, dropout=DROPOUT):
        super().__init__()
        layers = []
        for i, out_ch in enumerate(num_channels):
            in_ch = num_inputs if i == 0 else num_channels[i - 1]
            layers.append(TCNBlock(in_ch, out_ch, kernel_size, stride=1, dilation=2**i, dropout=dropout))
        self.network = nn.Sequential(*layers)
    def forward(self, x):
        return self.network(x)

class GenerativeCounterfactualVAE(nn.Module):
    def __init__(self, num_features=NUM_FEATURES, seq_len=SEQ_LEN, latent_dim=LATENT_DIM, cond_dim=COND_DIM):
        super().__init__()
        self.seq_len = seq_len
        tcn_ch = TCN_CHANNELS
        self.encoder_tcn = TemporalConvNet(num_features, tcn_ch)
        tcn_flat = seq_len * tcn_ch[-1]
        self.fc_mu     = nn.Linear(tcn_flat, latent_dim)
        self.fc_logvar = nn.Linear(tcn_flat, latent_dim)
        self.decoder_fc = nn.Sequential(
            nn.Linear(latent_dim + cond_dim, DECODER_HIDDEN), nn.ReLU(),
            nn.Linear(DECODER_HIDDEN, tcn_flat), nn.ReLU(),
        )
        self.decoder_tcn = TemporalConvNet(tcn_ch[-1], [32, num_features])

    def encode(self, x):
        h = self.encoder_tcn(x).flatten(start_dim=1)
        return self.fc_mu(h), self.fc_logvar(h)

    @staticmethod
    def reparameterize(mu, logvar):
        return mu + torch.randn_like(mu) * torch.exp(0.5 * logvar)

    def decode(self, z, condition):
        h = self.decoder_fc(torch.cat([z, condition], dim=-1))
        h = h.view(h.size(0), TCN_CHANNELS[-1], SEQ_LEN)
        return self.decoder_tcn(h)

    def forward(self, x, condition):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        return self.decode(z, condition), mu, logvar

def vae_loss_function(recon_x, x, mu, logvar):
    recon = F.mse_loss(recon_x, x, reduction="sum")
    kld   = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    return recon + KLD_WEIGHT * kld


In [ ]:
# ─── TRAIN ───────────────────────────────────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

model = GenerativeCounterfactualVAE().to(device)
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
loader = get_dataloader()

model.train()
for epoch in range(1, EPOCHS + 1):
    epoch_loss = 0.0
    for batch in loader:
        x = batch.permute(0, 2, 1).to(device)
        cond = torch.tensor([BASELINE_CONDITION] * x.size(0), dtype=torch.float32, device=device)
        optimizer.zero_grad()
        recon, mu, logvar = model(x, cond)
        loss = vae_loss_function(recon, x, mu, logvar)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    print(f"  Epoch {epoch}/{EPOCHS}  avg_loss={epoch_loss / len(loader.dataset):.4f}")
print("Training complete.")


In [ ]:
# ─── GENERATE COUNTERFACTUALS ─────────────────────────────────────────────────
model.eval()
for name, spec in SCENARIOS.items():
    with torch.no_grad():
        z = torch.randn(1, LATENT_DIM, device=device)
        cond = torch.tensor([spec["condition"]], dtype=torch.float32, device=device)
        out = model.decode(z, cond)
        demand = out[:, :NUM_NODES, :].squeeze(0).permute(1, 0).cpu().numpy()
        print(f"[{name}] {spec['description']}  shape={demand.shape}")
        np.save(f"{name}.npy", demand)
        print(f"  -> saved {name}.npy")
print("All counterfactual scenarios generated.")
